In [1]:
import os
import re
import random
import math
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

seed = 42
random.seed(seed)
torch.manual_seed(seed)

VOCAB_SIZE = 10000
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
NUM_LAYERS = 1
BATCH_SIZE = 64
LR = 0.001
EPOCHS = 10
MAX_LEN = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
try:
    from torchtext.datasets import IMDB
    print("Using torchtext.IMDB loader (raw text).")
    train_iter = IMDB(split="train")
    test_iter = IMDB(split="test")
    # torchtext returns tuples (label, text), where label in {"pos","neg"}
    train_data = [(label, text) for (label, text) in train_iter]
    test_data = [(label, text) for (label, text) in test_iter]
except Exception as e:
    print("torchtext IMDB not available or failed:", e)
    print("Falling back to keras.datasets.imdb (integer sequences). We'll convert back to words.")
    try:
        from tensorflow.keras.datasets import imdb
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        print("Using keras.datasets.imdb fallback.")
        (x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
        # get word index mapping
        word_index = imdb.get_word_index()
        # invert mapping
        index_word = {idx+3: word for word, idx in word_index.items()}  # offset used by keras
        index_word[0] = "<PAD>"
        index_word[1] = "<START>"
        index_word[2] = "<UNK>"
        index_word[3] = "<UNUSED>"
        # convert integer sequences back to text
        def seqs_to_texts(seqs):
            texts = []
            for seq in seqs:
                words = [index_word.get(i, "<UNK>") for i in seq]
                texts.append(" ".join(words))
            return texts
        train_texts = seqs_to_texts(x_train)
        test_texts = seqs_to_texts(x_test)
        train_data = [("pos" if y==1 else "neg", txt) for y, txt in zip(y_train, train_texts)]
        test_data = [("pos" if y==1 else "neg", txt) for y, txt in zip(y_test, test_texts)]
    except Exception as e2:
        raise RuntimeError("Failed to load IMDB dataset via torchtext and keras. Please ensure internet access or provide dataset.") from e2

print("Train samples:", len(train_data), "Test samples:", len(test_data))

torchtext IMDB not available or failed: [WinError 127] The specified procedure could not be found
Falling back to keras.datasets.imdb (integer sequences). We'll convert back to words.
Using keras.datasets.imdb fallback.
Train samples: 25000 Test samples: 25000


In [3]:
def clean_and_tokenize(text):
    text = text.replace("\r\n", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    text = text.lower()
    tokens = re.findall(r"\w+|[^\s\w]", text, re.UNICODE)
    return tokens

print("Example tokens:", clean_and_tokenize(train_data[0][1])[:20])

Example tokens: ['<', 'start', '>', 'this', 'film', 'was', 'just', 'brilliant', 'casting', 'location', 'scenery', 'story', 'direction', 'everyone', "'", 's', 'really', 'suited', 'the', 'part']


In [4]:
all_train_tokens = []
for label, text in train_data:
    toks = clean_and_tokenize(text)
    all_train_tokens.extend(toks)

counter = Counter(all_train_tokens)
most_common = counter.most_common(VOCAB_SIZE)
vocab_tokens = [tok for tok, _ in most_common]

PAD = "<PAD>"
UNK = "<UNK>"

itos = [PAD, UNK] + vocab_tokens
stoi = {w:i for i,w in enumerate(itos)}
vocab_size = len(itos)
print("Final vocabulary size (with special tokens):", vocab_size)

Final vocabulary size (with special tokens): 9817


In [5]:
def text_to_sequence(text, stoi, max_len=MAX_LEN):
    toks = clean_and_tokenize(text)
    idxs = [stoi.get(t, stoi[UNK]) for t in toks]
    if len(idxs) > max_len:
        idxs = idxs[:max_len]
    if len(idxs) < max_len:
        idxs = idxs + [stoi[PAD]] * (max_len - len(idxs))
    return idxs

train_texts_idx = [text_to_sequence(txt, stoi) for _, txt in train_data]
train_labels = [1 if label=="pos" else 0 for label, _ in train_data]

test_texts_idx = [text_to_sequence(txt, stoi) for _, txt in test_data]
test_labels = [1 if label=="pos" else 0 for label, _ in test_data]

print("Example sequence length:", len(train_texts_idx[0]))

Example sequence length: 256


In [6]:
class IMDBDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        x = torch.tensor(self.sequences[idx], dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.float)  # float for BCEWithLogits
        return x, y

train_dataset = IMDBDataset(train_texts_idx, train_labels)
test_dataset = IMDBDataset(test_texts_idx, test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [7]:
class RNNClassifier(nn.Module):
    def __init__(self, rnn_type="RNN", vocab_size=vocab_size, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        rnn_type_upper = rnn_type.upper()
        if rnn_type_upper == "RNN":
            self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        elif rnn_type_upper == "LSTM":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        elif rnn_type_upper == "GRU":
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        else:
            raise ValueError("Unsupported rnn_type")
        self.fc = nn.Linear(hidden_dim, 1)  # binary output (logit)

        self.rnn_type = rnn_type_upper

    def forward(self, x):
        emb = self.embedding(x)  # (batch, seq_len, emb)
        out, hidden = self.rnn(emb)  # out: (batch, seq_len, hidden)

        last = out[:, -1, :]  # (batch, hidden)
        logits = self.fc(last).squeeze(1)  # (batch,)
        return logits

_test_model = RNNClassifier("RNN").to(device)
print(_test_model)

RNNClassifier(
  (embedding): Embedding(9817, 100, padding_idx=0)
  (rnn): RNN(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)


In [8]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    preds = []
    truths = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            batch_preds = (probs >= 0.5).astype(int).tolist()
            preds.extend(batch_preds)
            truths.extend(yb.numpy().astype(int).tolist())
    acc = accuracy_score(truths, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(truths, preds, average="binary", zero_division=0)
    return acc, precision, recall, f1

In [9]:
results = {}

for rnn_type in ["RNN", "LSTM", "GRU"]:
    print("\n\n=== Training model:", rnn_type, "===")
    model = RNNClassifier(rnn_type=rnn_type,
                          vocab_size=vocab_size,
                          embedding_dim=EMBEDDING_DIM,
                          hidden_dim=HIDDEN_DIM,
                          num_layers=NUM_LAYERS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        acc, prec, rec, f1 = evaluate(model, test_loader, device)
        print(f"[{rnn_type}] Epoch {epoch}/{EPOCHS} — train_loss: {loss:.4f}  test_acc: {acc:.4f}  prec: {prec:.4f}  rec: {rec:.4f}  f1: {f1:.4f}")
    # final metrics
    acc, prec, rec, f1 = evaluate(model, test_loader, device)
    results[rnn_type] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}
    # save model
    torch.save({
        "model_state_dict": model.state_dict(),
        "vocab": itos,
        "config": {
            "rnn_type": rnn_type,
            "vocab_size": vocab_size,
            "embedding_dim": EMBEDDING_DIM,
            "hidden_dim": HIDDEN_DIM,
            "num_layers": NUM_LAYERS,
            "max_len": MAX_LEN
        }
    }, f"imdb_{rnn_type.lower()}_model.pth")
    print(f"Saved checkpoint -> imdb_{rnn_type.lower()}_model.pth")

print("\n=== Summary ===")
for k,v in results.items():
    print(k, v)



=== Training model: RNN ===
[RNN] Epoch 1/10 — train_loss: 0.6963  test_acc: 0.5052  prec: 0.5032  rec: 0.8259  f1: 0.6254
[RNN] Epoch 2/10 — train_loss: 0.6908  test_acc: 0.5032  prec: 0.5086  rec: 0.1858  f1: 0.2722
[RNN] Epoch 3/10 — train_loss: 0.6900  test_acc: 0.5016  prec: 0.5020  rec: 0.4102  f1: 0.4514
[RNN] Epoch 4/10 — train_loss: 0.6938  test_acc: 0.4996  prec: 0.4994  rec: 0.3118  f1: 0.3839
[RNN] Epoch 5/10 — train_loss: 0.6919  test_acc: 0.5084  prec: 0.5052  rec: 0.8219  f1: 0.6258
[RNN] Epoch 6/10 — train_loss: 0.6877  test_acc: 0.5040  prec: 0.5213  rec: 0.0978  f1: 0.1646
[RNN] Epoch 7/10 — train_loss: 0.6818  test_acc: 0.5080  prec: 0.5047  rec: 0.8630  f1: 0.6369
[RNN] Epoch 8/10 — train_loss: 0.6744  test_acc: 0.5083  prec: 0.5313  rec: 0.1411  f1: 0.2230
[RNN] Epoch 9/10 — train_loss: 0.6624  test_acc: 0.5113  prec: 0.5365  rec: 0.1658  f1: 0.2534
[RNN] Epoch 10/10 — train_loss: 0.6469  test_acc: 0.5236  prec: 0.5148  rec: 0.8202  f1: 0.6326
Saved checkpoint ->

In [10]:
print("Final evaluation on test set")
for model_name, metrics in results.items():
    print(f"{model_name} — Acc: {metrics['accuracy']:.4f}  Prec: {metrics['precision']:.4f}  Rec: {metrics['recall']:.4f}  F1: {metrics['f1']:.4f}")

Final evaluation on test set:
RNN — Acc: 0.5236  Prec: 0.5148  Rec: 0.8202  F1: 0.6326
LSTM — Acc: 0.7058  Prec: 0.6760  Rec: 0.7907  F1: 0.7289
GRU — Acc: 0.8527  Prec: 0.8468  Rec: 0.8612  F1: 0.8540
